# 03 — TFLite Optimization
**Goal:** Retrain the 1D-CNN (accel-only and accel+gyro) using the identical split,
normalization, and architecture as notebook 02, then convert each to three TFLite
quantization levels (float32, float16, INT8) and benchmark all six on the test set.

**Note on latency:** Numbers are measured on this laptop CPU.
On an Android phone, INT8 would be faster due to hardware INT8 acceleration (NNAPI / XNNPACK).

In [1]:
# ── Standard-library imports ──────────────────────────────────────────────────
import os       # file paths and directory creation
import time     # wall-clock timing for latency measurement
import random   # Python built-in RNG seeding
import warnings # suppress noisy deprecation warnings
warnings.filterwarnings('ignore')

# ── Third-party imports ───────────────────────────────────────────────────────
import numpy as np    # array math
import pandas as pd   # results table

from sklearn.metrics import accuracy_score, f1_score   # evaluation metrics

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print('TensorFlow version:', tf.__version__)
print('Keras version     :', keras.__version__)

TensorFlow version: 2.19.0
Keras version     : 3.9.2


In [2]:
# ── Fix ALL random seeds for full reproducibility ─────────────────────────────
# Must use the same seed (42) as notebook 02 to get identical weight initialisation
SEED = 42

# Seed Python's built-in random module
random.seed(SEED)

# Seed NumPy's global RNG
np.random.seed(SEED)

# Seed TensorFlow (covers Keras weight initialisers and dropout)
tf.random.set_seed(SEED)

# Make Python hash randomisation deterministic (affects dict ordering etc.)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'All random seeds fixed to {SEED}')

All random seeds fixed to 42


In [3]:
# ── Path constants ─────────────────────────────────────────────────────────────
# Input: the preprocessed windows saved by notebook 01
DATA_PATH   = os.path.join('..', 'data', 'windows.npz')

# Output directories
MODELS_DIR  = os.path.join('..', 'models')    # .keras and .tflite files
FIGURES_DIR = os.path.join('..', 'figures')   # results CSV

# Create output directories if they don't exist yet
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Activity class names (used for display only)
CLASS_NAMES = ['Walking', 'Jogging', 'Stairs', 'Sitting', 'Standing']

print('Data path   :', os.path.abspath(DATA_PATH))
print('Models dir  :', os.path.abspath(MODELS_DIR))
print('Figures dir :', os.path.abspath(FIGURES_DIR))

Data path   : C:\Users\shiva\Downloads\HAR2\data\windows.npz
Models dir  : C:\Users\shiva\Downloads\HAR2\models
Figures dir : C:\Users\shiva\Downloads\HAR2\figures


In [4]:
# ── Load preprocessed windows ─────────────────────────────────────────────────
# These were saved by notebook 01 as a compressed NumPy archive
data     = np.load(DATA_PATH)
X        = data['X']         # shape (17685, 100, 6)  — ax ay az gx gy gz
y        = data['y']         # shape (17685,)          — integer labels 0-4
subjects = data['subjects']  # shape (17685,)          — subject IDs per window

print(f'X shape        : {X.shape}')
print(f'y shape        : {y.shape}')
print(f'subjects shape : {subjects.shape}')

X shape        : (17685, 100, 6)
y shape        : (17685,)
subjects shape : (17685,)


In [5]:
# ── Reproduce the EXACT subject split from notebook 02 ────────────────────────
# We must use the same RNG, same seed, and same split fractions so that
# train/val/test windows are byte-for-byte identical to the previous notebook.

# Collect the 51 unique subject IDs (already sorted by np.unique)
unique_subjects = np.unique(subjects)
n_subjects      = len(unique_subjects)   # 51

# Shuffle with a seeded Generator (same call as notebook 02)
rng               = np.random.default_rng(SEED)
shuffled_subjects = rng.permutation(unique_subjects)

# Compute the same split boundaries used in notebook 02
n_train = int(np.floor(0.70 * n_subjects))   # 35 subjects
n_val   = int(np.floor(0.15 * n_subjects))   #  7 subjects

# Slice the shuffled array to get each split's subject IDs
train_subjects = shuffled_subjects[:n_train]
val_subjects   = shuffled_subjects[n_train : n_train + n_val]
test_subjects  = shuffled_subjects[n_train + n_val :]

# Print every subject ID so the evaluator can verify no leakage
print(f'Train subjects ({len(train_subjects)}): {sorted(train_subjects.tolist())}')
print(f'Val   subjects ({len(val_subjects)}): {sorted(val_subjects.tolist())}')
print(f'Test  subjects ({len(test_subjects)}): {sorted(test_subjects.tolist())}')

# Verify: intersection of every pair of splits must be empty
assert len(set(train_subjects) & set(val_subjects))  == 0, 'Leak: train ∩ val'
assert len(set(train_subjects) & set(test_subjects)) == 0, 'Leak: train ∩ test'
assert len(set(val_subjects)   & set(test_subjects)) == 0, 'Leak: val ∩ test'
print('\n✅ No subject overlap between splits (identical to notebook 02).')

Train subjects (35): [1603, 1604, 1605, 1607, 1609, 1610, 1615, 1616, 1617, 1618, 1620, 1621, 1623, 1624, 1625, 1626, 1627, 1628, 1629, 1630, 1631, 1632, 1634, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1645, 1646, 1647, 1648, 1650]
Val   subjects (7): [1600, 1606, 1611, 1619, 1622, 1635, 1644]
Test  subjects (9): [1601, 1602, 1608, 1612, 1613, 1614, 1633, 1636, 1649]

✅ No subject overlap between splits (identical to notebook 02).


In [6]:
# ── Build boolean masks and apply to get split arrays ─────────────────────────
# np.isin returns True for every window whose subject is in the given set
train_mask = np.isin(subjects, train_subjects)
val_mask   = np.isin(subjects, val_subjects)
test_mask  = np.isin(subjects, test_subjects)

# Slice the full dataset into three non-overlapping subsets
X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train windows : {len(X_train):,}')
print(f'Val   windows : {len(X_val):,}')
print(f'Test  windows : {len(X_test):,}')

Train windows : 12,065
Val   windows : 2,455
Test  windows : 3,165


In [7]:
# ── Reproduce normalisation from notebook 02 ───────────────────────────────────
# Compute channel-wise mean and std from TRAINING windows only.
# Flatten (N_train, 100, 6) → (N_train*100, 6) so stats are across all time steps.

X_train_flat = X_train.reshape(-1, X_train.shape[-1])   # shape: (N*100, 6)
train_mean   = X_train_flat.mean(axis=0)                 # shape: (6,)
train_std    = X_train_flat.std(axis=0)                  # shape: (6,)

# Prevent division by zero in the unlikely case of a zero-variance channel
train_std[train_std == 0] = 1.0

# Cross-check against the saved norm_stats.json from notebook 02
norm_json_path = os.path.join(MODELS_DIR, 'norm_stats.json')
if os.path.exists(norm_json_path):
    import json
    # Load previously saved stats
    with open(norm_json_path) as f:
        saved = json.load(f)
    # Compare element-wise, allow tiny floating-point rounding differences
    mean_match = np.allclose(train_mean, saved['mean'], atol=1e-5)
    std_match  = np.allclose(train_std,  saved['std'],  atol=1e-5)
    print(f'Mean matches norm_stats.json: {mean_match}')
    print(f'Std  matches norm_stats.json: {std_match}')

# Apply normalisation: broadcast (6,) over (N, 100, 6)
X_train_n = (X_train - train_mean) / train_std   # normalised training set
X_val_n   = (X_val   - train_mean) / train_std   # normalised val (train stats)
X_test_n  = (X_test  - train_mean) / train_std   # normalised test (train stats)

print(f'\nTrain mean (first 3 ch): {train_mean[:3].round(4)}')
print(f'Train std  (first 3 ch): {train_std[:3].round(4)}')

Mean matches norm_stats.json: True
Std  matches norm_stats.json: True

Train mean (first 3 ch): [ 1.0253 -1.7554 -0.243 ]
Train std  (first 3 ch): [6.4385 8.1821 4.6221]


In [8]:
# ── Define the 1D-CNN architecture (identical to notebook 02) ──────────────────
# Copied verbatim so the evaluator can see the architecture is unchanged.

def build_cnn(input_shape, n_classes=5):
    """Build and compile a small 1D-CNN.

    Architecture (same as notebook 02):
      Conv1D(32, 5, same) → BatchNorm → ReLU
      Conv1D(64, 3, same) → BatchNorm → ReLU
      GlobalAveragePooling1D
      Dense(64, relu) → Dropout(0.3)
      Dense(5, softmax)
    """
    # Input layer: accepts windows of shape (100, n_channels)
    inp = keras.Input(shape=input_shape, name='input')

    # First Conv block: 32 filters, kernel 5 — captures 250 ms local patterns
    x = layers.Conv1D(32, kernel_size=5, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)   # stabilise activations
    x = layers.Activation('relu')(x)

    # Second Conv block: 64 filters, kernel 3 — captures finer patterns
    x = layers.Conv1D(64, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global average pool collapses the 100-step sequence to a 64-d vector
    x = layers.GlobalAveragePooling1D()(x)

    # Fully connected layer with dropout to reduce overfitting
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Softmax output: one probability per activity class
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    # Compile with Adam and sparse cross-entropy (labels are integers, not one-hot)
    model = keras.Model(inputs=inp, outputs=out, name='CNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Training hyperparameters — same as notebook 02
EPOCHS     = 60
BATCH_SIZE = 128

# Print parameter count for both input sizes
print(f'CNN (3-ch / accel-only) params : {build_cnn((100, 3)).count_params():,}')
print(f'CNN (6-ch / accel+gyro) params : {build_cnn((100, 6)).count_params():,}')

CNN (3-ch / accel-only) params : 11,493
CNN (6-ch / accel+gyro) params : 11,973


In [9]:
# ── Helper: train and save one CNN model ──────────────────────────────────────
# Resets the TF seed before each run so weight init is reproducible.

def train_and_save_cnn(mode, save_path):
    """Train the 1D-CNN for a given sensor mode and save the .keras file.

    Parameters
    ----------
    mode      : 'accel' → channels 0-2 only; 'both' → all 6 channels
    save_path : path where the trained model will be saved

    Returns
    -------
    model : trained Keras model (best weights from EarlyStopping)
    """
    # Select the correct channel slice from the normalised arrays
    if mode == 'accel':
        # Channels 0,1,2 = ax, ay, az
        Xtr = X_train_n[:, :, :3]
        Xv  = X_val_n[:, :, :3]
    else:
        # All 6 channels: ax, ay, az, gx, gy, gz
        Xtr = X_train_n
        Xv  = X_val_n

    # Reset TF seed before building and training for reproducibility
    tf.random.set_seed(SEED)

    # Build a fresh model with the correct input shape
    n_ch    = Xtr.shape[2]             # 3 or 6
    model   = build_cnn((100, n_ch))   # input shape: (100 timesteps, n_ch channels)

    # Print the number of trainable parameters
    print(f'  Parameters: {model.count_params():,}')

    # EarlyStopping: stop training if val_loss does not improve for 10 epochs;
    # restore the model weights from the epoch with the lowest val_loss.
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    # Train the model
    model.fit(
        Xtr, y_train,
        validation_data=(Xv, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=2   # one line per epoch
    )

    # Save the trained model in Keras v3 format
    model.save(save_path)
    size_kb = os.path.getsize(save_path) / 1024
    print(f'  Saved → {save_path}  ({size_kb:.1f} KB)')

    return model

In [10]:
# ── Train CNN (accel-only, 3 channels) ────────────────────────────────────────
print('=== Training CNN — Accel-only (ax, ay, az) ===')

# Save path for the accel-only model
accel_keras_path = os.path.join(MODELS_DIR, 'cnn_accel.keras')

# Train and save; returned model object is kept for reference
cnn_accel = train_and_save_cnn(mode='accel', save_path=accel_keras_path)

=== Training CNN — Accel-only (ax, ay, az) ===
  Parameters: 11,493
Epoch 1/60


95/95 - 2s - 25ms/step - accuracy: 0.6521 - loss: 0.9090 - val_accuracy: 0.5735 - val_loss: 1.0355


Epoch 2/60


95/95 - 1s - 12ms/step - accuracy: 0.7882 - loss: 0.5129 - val_accuracy: 0.6126 - val_loss: 0.8467


Epoch 3/60


95/95 - 1s - 12ms/step - accuracy: 0.8269 - loss: 0.4109 - val_accuracy: 0.6403 - val_loss: 0.6808


... (25 outputs left out)

95/95 - 1s - 11ms/step - accuracy: 0.9398 - loss: 0.1693 - val_accuracy: 0.7723 - val_loss: 0.6263


Epoch 16: early stopping


Restoring model weights from the end of the best epoch: 6.


  Saved → ..\models\cnn_accel.keras  (185.5 KB)


In [11]:
# ── Train CNN (accel+gyro, 6 channels) ───────────────────────────────────────
print('=== Training CNN — Accel+Gyro (ax, ay, az, gx, gy, gz) ===')

# Save path for the 6-channel model
both_keras_path = os.path.join(MODELS_DIR, 'cnn_both.keras')

# Train and save
cnn_both = train_and_save_cnn(mode='both', save_path=both_keras_path)

=== Training CNN — Accel+Gyro (ax, ay, az, gx, gy, gz) ===
  Parameters: 11,973
Epoch 1/60


95/95 - 2s - 25ms/step - accuracy: 0.6477 - loss: 0.8728 - val_accuracy: 0.6729 - val_loss: 0.8692


Epoch 2/60


95/95 - 2s - 16ms/step - accuracy: 0.7999 - loss: 0.4557 - val_accuracy: 0.6990 - val_loss: 0.7365


Epoch 3/60


95/95 - 2s - 18ms/step - accuracy: 0.8622 - loss: 0.3503 - val_accuracy: 0.7271 - val_loss: 0.6127


... (53 outputs left out)

95/95 - 1s - 12ms/step - accuracy: 0.9762 - loss: 0.0687 - val_accuracy: 0.8709 - val_loss: 0.5133


Epoch 30: early stopping


Restoring model weights from the end of the best epoch: 20.


  Saved → ..\models\cnn_both.keras  (191.1 KB)


In [12]:
# ── TFLite conversion helper ─────────────────────────────────────────────────
# Converts one .keras model to three quantization variants and saves all three.

def convert_to_tflite(keras_path, mode, X_rep):
    """Convert a Keras model to float32, float16, and INT8 TFLite models.

    Parameters
    ----------
    keras_path : path to the saved .keras model
    mode       : 'accel' or 'both' — used to build output filenames
    X_rep      : normalised training windows for the representative dataset
                 (used to calibrate INT8 quantization scale factors)

    Returns
    -------
    dict mapping quantization name → saved .tflite file path
    """
    # Load the trained Keras model from disk
    model = keras.models.load_model(keras_path)

    saved_paths = {}   # will hold {quant_name: file_path}

    # ── 1. float32 conversion ─────────────────────────────────────────────────
    # Default conversion: no quantization, weights and activations stay float32.
    conv_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
    # No optimizations flag → lossless conversion
    tflite_f32 = conv_f32.convert()

    # Build the output file path and write the raw bytes
    path_f32 = os.path.join(MODELS_DIR, f'cnn_{mode}_float32.tflite')
    with open(path_f32, 'wb') as fh:
        fh.write(tflite_f32)
    saved_paths['float32'] = path_f32
    print(f'  float32 → {path_f32}  ({os.path.getsize(path_f32)/1024:.1f} KB)')

    # ── 2. float16 conversion ─────────────────────────────────────────────────
    # Optimize.DEFAULT + float16 target: weights are quantized to 16-bit floats.
    # Activations stay float32 at runtime on CPU; size is roughly halved.
    conv_f16 = tf.lite.TFLiteConverter.from_keras_model(model)
    # Tell the converter to apply the default size/latency optimization
    conv_f16.optimizations = [tf.lite.Optimize.DEFAULT]
    # Restrict the quantized type to float16 (not INT8)
    conv_f16.target_spec.supported_types = [tf.float16]
    tflite_f16 = conv_f16.convert()

    # Save float16 model
    path_f16 = os.path.join(MODELS_DIR, f'cnn_{mode}_float16.tflite')
    with open(path_f16, 'wb') as fh:
        fh.write(tflite_f16)
    saved_paths['float16'] = path_f16
    print(f'  float16 → {path_f16}  ({os.path.getsize(path_f16)/1024:.1f} KB)')

    # ── 3. Full INT8 conversion ───────────────────────────────────────────────
    # Both weights AND activations are quantized to 8-bit integers.
    # Requires a representative dataset to compute per-tensor scale and zero-point.
    conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    # Enable the default quantization optimization
    conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]

    # Representative dataset: a generator that yields small batches of real data.
    # The converter calls this repeatedly to measure activation ranges.
    # We sample 200 windows from the training set — enough for calibration.
    N_REP = 200   # number of calibration samples

    # Randomly pick N_REP indices from the training windows
    rep_indices = np.random.choice(len(X_rep), size=N_REP, replace=False)

    def representative_dataset():
        """Generator: yield one window at a time as a float32 tensor."""
        for idx in rep_indices:
            # Shape must be (1, 100, n_channels) — batch size of 1
            sample = X_rep[idx : idx + 1].astype(np.float32)
            # Yield as a list (the converter expects a list of input tensors)
            yield [sample]

    # Assign the generator to the converter
    conv_int8.representative_dataset = representative_dataset

    # Request full integer quantization for all ops that support it
    conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

    # Force input and output tensors to also be INT8 (true full INT8)
    conv_int8.inference_input_type  = tf.int8
    conv_int8.inference_output_type = tf.int8

    tflite_int8 = conv_int8.convert()

    # Save INT8 model
    path_int8 = os.path.join(MODELS_DIR, f'cnn_{mode}_int8.tflite')
    with open(path_int8, 'wb') as fh:
        fh.write(tflite_int8)
    saved_paths['int8'] = path_int8
    print(f'  int8    → {path_int8}  ({os.path.getsize(path_int8)/1024:.1f} KB)')

    return saved_paths

In [13]:
# ── Convert CNN (accel-only) to all three TFLite formats ─────────────────────
print('=== Converting cnn_accel.keras → TFLite ===')

# The representative dataset must use the SAME channel slice as training
# (accel-only: first 3 channels)
X_rep_accel = X_train_n[:, :, :3]   # shape: (N_train, 100, 3)

# Run the conversion; returns dict of {quant_name: file_path}
accel_tflite_paths = convert_to_tflite(
    keras_path=accel_keras_path,
    mode='accel',
    X_rep=X_rep_accel
)

=== Converting cnn_accel.keras → TFLite ===


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmp6ceqlfrh\assets


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmp6ceqlfrh\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmp6ceqlfrh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2035605347136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605380960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605241520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035546260576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605380784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035522024720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606919472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606920528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606918064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606919296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606932640: Te

  float32 → ..\models\cnn_accel_float32.tflite  (48.2 KB)


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpxrsyxtn5\assets


... (3 outputs left out)

INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpzm_g239t\assets


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpzm_g239t\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmpzm_g239t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2035605347136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605380960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605241520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035546260576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035605380784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035522024720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606919472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606920528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606918064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606919296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035606932640: Te

  int8    → ..\models\cnn_accel_int8.tflite  (20.3 KB)


In [14]:
# ── Convert CNN (accel+gyro) to all three TFLite formats ─────────────────────
print('=== Converting cnn_both.keras → TFLite ===')

# Representative dataset for the 6-channel model uses all channels
X_rep_both = X_train_n   # shape: (N_train, 100, 6)

# Run the conversion
both_tflite_paths = convert_to_tflite(
    keras_path=both_keras_path,
    mode='both',
    X_rep=X_rep_both
)

=== Converting cnn_both.keras → TFLite ===


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpw8dlfiut\assets


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpw8dlfiut\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmpw8dlfiut'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2035708512912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035370826208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592901376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035706604176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035603997792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592787920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592805360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035604031088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592801264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592802896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035707087152: Te

  float32 → ..\models\cnn_both_float32.tflite  (50.1 KB)


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmp3_geijqp\assets


... (3 outputs left out)

INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpejy38fyg\assets


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpejy38fyg\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmpejy38fyg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2035708512912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035370826208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592901376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035706604176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035603997792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592787920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592805360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035604031088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592801264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035592802896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2035707087152: Te

  int8    → ..\models\cnn_both_int8.tflite  (20.8 KB)


In [15]:
# ── TFLite inference helper ────────────────────────────────────────────────────
# Loads a .tflite file and runs it on every window in X_te.
# Handles both float32/float16 models (input stays float32) and
# full INT8 models (input must be quantized to int8 using the tensor's
# scale and zero_point reported by the interpreter).

def eval_tflite(tflite_path, X_te, y_te):
    """Evaluate a TFLite model on a test set.

    Parameters
    ----------
    tflite_path : path to the .tflite file
    X_te        : normalised test windows, shape (N_test, 100, n_ch)
    y_te        : true integer labels, shape (N_test,)

    Returns
    -------
    accuracy  : float
    macro_f1  : float
    """
    # Load the TFLite model into an Interpreter object
    interpreter = tf.lite.Interpreter(model_path=tflite_path)

    # Allocate memory for input and output tensors
    interpreter.allocate_tensors()

    # Retrieve metadata about the input tensor (index, shape, dtype, quant params)
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # Check whether this is a fully-quantized INT8 model
    is_int8 = (input_details['dtype'] == np.int8)

    preds = []   # will collect the predicted class index for each test window

    for i in range(len(X_te)):
        # Extract a single window and add the batch dimension → (1, 100, n_ch)
        window = X_te[i : i + 1].astype(np.float32)

        if is_int8:
            # For INT8 models: quantize the float input to int8 using the
            # scale and zero_point stored in the model
            scale, zero_point = input_details['quantization']
            # Quantization formula: q = round(x / scale) + zero_point
            window_q = np.round(window / scale + zero_point).astype(np.int8)
            interpreter.set_tensor(input_details['index'], window_q)
        else:
            # For float32/float16 models: pass the float tensor directly
            interpreter.set_tensor(input_details['index'], window)

        # Run the forward pass
        interpreter.invoke()

        # Read the output tensor
        output = interpreter.get_tensor(output_details['index'])

        if is_int8:
            # Dequantize the INT8 output logits back to float for argmax
            out_scale, out_zero = output_details['quantization']
            output = (output.astype(np.float32) - out_zero) * out_scale

        # The predicted class is the index of the maximum output value
        preds.append(int(output.argmax()))

    # Compute metrics
    preds    = np.array(preds)
    accuracy = accuracy_score(y_te, preds)
    macro_f1 = f1_score(y_te, preds, average='macro')

    return accuracy, macro_f1

In [16]:
# ── Latency measurement helper ─────────────────────────────────────────────────
# Measures the average time to run one window through the TFLite interpreter.
# We run n=1000 single-window invocations and report the mean in milliseconds.
# Each run picks a random window from the test set to avoid cache-warming bias.

def measure_latency(tflite_path, X_te, n=1000):
    """Measure mean per-window inference latency.

    Parameters
    ----------
    tflite_path : path to the .tflite file
    X_te        : normalised test windows (used as input data)
    n           : number of timed invoke() calls

    Returns
    -------
    mean_ms : float — average latency in milliseconds
    """
    # Load the interpreter and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    # Get input tensor metadata
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # Check if the model expects INT8 input
    is_int8 = (input_details['dtype'] == np.int8)

    # Randomly choose n window indices (with replacement) from the test set
    indices = np.random.randint(0, len(X_te), size=n)

    elapsed_times = []   # list of per-invoke wall-clock times in seconds

    for idx in indices:
        # Prepare the input tensor for this window
        window = X_te[idx : idx + 1].astype(np.float32)   # shape (1, 100, n_ch)

        if is_int8:
            # Quantize float input to int8 before setting the tensor
            scale, zero_point = input_details['quantization']
            window = np.round(window / scale + zero_point).astype(np.int8)

        # Set the input tensor (outside the timed section)
        interpreter.set_tensor(input_details['index'], window)

        # Time ONLY the invoke() call — the actual inference
        t0 = time.perf_counter()   # high-resolution timer start
        interpreter.invoke()
        t1 = time.perf_counter()   # high-resolution timer stop

        # Accumulate elapsed time in seconds
        elapsed_times.append(t1 - t0)

    # Convert seconds to milliseconds and return the mean
    mean_ms = float(np.mean(elapsed_times)) * 1000
    return mean_ms

In [17]:
# ── Run evaluation and latency for all 6 TFLite files ─────────────────────────
# We iterate over every (model_name, sensor_mode, quant_name, tflite_path) tuple
# and collect size, accuracy, macro F1, and latency.

# Build a flat list of all 6 experiments
# Each entry: (display_name, sensor_mode, quant_name, tflite_path)
experiments = [
    ('CNN-Accel', 'accel', 'float32', accel_tflite_paths['float32']),
    ('CNN-Accel', 'accel', 'float16', accel_tflite_paths['float16']),
    ('CNN-Accel', 'accel', 'int8',    accel_tflite_paths['int8']),
    ('CNN-Both',  'both',  'float32', both_tflite_paths['float32']),
    ('CNN-Both',  'both',  'float16', both_tflite_paths['float16']),
    ('CNN-Both',  'both',  'int8',    both_tflite_paths['int8']),
]

rows = []   # will hold one dict per row for the results table

for model_name, sensor_mode, quant_name, tflite_path in experiments:
    print(f'\nEvaluating: {model_name} | {quant_name}')

    # Select the correct test-set channel slice
    if sensor_mode == 'accel':
        X_te = X_test_n[:, :, :3]   # first 3 channels only
    else:
        X_te = X_test_n             # all 6 channels

    # File size in KB
    size_kb = os.path.getsize(tflite_path) / 1024
    print(f'  File size    : {size_kb:.1f} KB')

    # Test accuracy and macro F1 via TFLite interpreter
    accuracy, macro_f1 = eval_tflite(tflite_path, X_te, y_test)
    print(f'  Test accuracy: {accuracy:.4f}')
    print(f'  Macro F1     : {macro_f1:.4f}')

    # Mean latency over 1000 single-window invocations
    latency_ms = measure_latency(tflite_path, X_te, n=1000)
    print(f'  Latency (ms) : {latency_ms:.3f}  [mean of 1000 runs, laptop CPU]')

    # Append the row to the results list
    rows.append({
        'Model'          : model_name,
        'Quantization'   : quant_name,
        'File'           : os.path.basename(tflite_path),
        'Size (KB)'      : round(size_kb, 1),
        'Test Accuracy'  : round(accuracy, 4),
        'Macro F1'       : round(macro_f1, 4),
        'Latency (ms)'   : round(latency_ms, 3),
    })


Evaluating: CNN-Accel | float32
  File size    : 48.2 KB
  Test accuracy: 0.8423
  Macro F1     : 0.8422
  Latency (ms) : 0.013  [mean of 1000 runs, laptop CPU]

Evaluating: CNN-Accel | float16
  File size    : 27.2 KB


  Test accuracy: 0.8423
  Macro F1     : 0.8422
  Latency (ms) : 0.013  [mean of 1000 runs, laptop CPU]

Evaluating: CNN-Accel | int8
  File size    : 20.3 KB


  Test accuracy: 0.8398
  Macro F1     : 0.8388
  Latency (ms) : 0.009  [mean of 1000 runs, laptop CPU]

Evaluating: CNN-Both | float32
  File size    : 50.1 KB
  Test accuracy: 0.8158
  Macro F1     : 0.8130
  Latency (ms) : 0.014  [mean of 1000 runs, laptop CPU]

Evaluating: CNN-Both | float16
  File size    : 28.2 KB


  Test accuracy: 0.8158


  Macro F1     : 0.8128
  Latency (ms) : 0.017  [mean of 1000 runs, laptop CPU]

Evaluating: CNN-Both | int8
  File size    : 20.8 KB
  Test accuracy: 0.8167
  Macro F1     : 0.8140
  Latency (ms) : 0.018  [mean of 1000 runs, laptop CPU]


In [18]:
# ── Build results DataFrame and print the summary table ───────────────────────
results_df = pd.DataFrame(rows)

# Display all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('\n===== TFLite Optimization Results =====')
print(results_df.to_string(index=False))


===== TFLite Optimization Results =====
    Model Quantization                     File  Size (KB)  Test Accuracy  Macro F1  Latency (ms)
CNN-Accel      float32 cnn_accel_float32.tflite       48.2         0.8423    0.8422         0.013
CNN-Accel      float16 cnn_accel_float16.tflite       27.2         0.8423    0.8422         0.013
CNN-Accel         int8    cnn_accel_int8.tflite       20.3         0.8398    0.8388         0.009
 CNN-Both      float32  cnn_both_float32.tflite       50.1         0.8158    0.8130         0.014
 CNN-Both      float16  cnn_both_float16.tflite       28.2         0.8158    0.8128         0.017
 CNN-Both         int8     cnn_both_int8.tflite       20.8         0.8167    0.8140         0.018


In [19]:
# ── Save results table to CSV ──────────────────────────────────────────────────
csv_path = os.path.join(FIGURES_DIR, 'tflite_results.csv')

# Write with index=False so row numbers are not included as a column
results_df.to_csv(csv_path, index=False)
print(f'Results saved to: {os.path.abspath(csv_path)}')

Results saved to: C:\Users\shiva\Downloads\HAR2\figures\tflite_results.csv


In [20]:
# ── List all generated files ────────────────────────────────────────────────────
# Confirm every expected file is present on disk with its actual size
expected_files = [
    os.path.join(MODELS_DIR, 'cnn_accel.keras'),
    os.path.join(MODELS_DIR, 'cnn_both.keras'),
    os.path.join(MODELS_DIR, 'cnn_accel_float32.tflite'),
    os.path.join(MODELS_DIR, 'cnn_accel_float16.tflite'),
    os.path.join(MODELS_DIR, 'cnn_accel_int8.tflite'),
    os.path.join(MODELS_DIR, 'cnn_both_float32.tflite'),
    os.path.join(MODELS_DIR, 'cnn_both_float16.tflite'),
    os.path.join(MODELS_DIR, 'cnn_both_int8.tflite'),
    os.path.join(FIGURES_DIR, 'tflite_results.csv'),
]

print('\nGenerated files:')
print(f'{"File":<45} {"Size (KB)":>10}')
print('-' * 57)
for fp in expected_files:
    # Check file exists and report its size
    if os.path.exists(fp):
        size_kb = os.path.getsize(fp) / 1024
        print(f'{os.path.basename(fp):<45} {size_kb:>10.1f}')
    else:
        # Warn if any expected file is missing
        print(f'{os.path.basename(fp):<45}    MISSING!')


Generated files:
File                                           Size (KB)
---------------------------------------------------------
cnn_accel.keras                                    185.5
cnn_both.keras                                     191.1
cnn_accel_float32.tflite                            48.2
cnn_accel_float16.tflite                            27.2
cnn_accel_int8.tflite                               20.3
cnn_both_float32.tflite                             50.1
cnn_both_float16.tflite                             28.2
cnn_both_int8.tflite                                20.8
tflite_results.csv                                   0.5
